# Project: A SQL Assistant with On-Demand Skills

This project uses **progressive disclosure** -- loading information on demand instead of
upfront -- to implement **skills**: specialized, prompt-based instructions the agent loads via
tool calls, rather than by dynamically swapping the whole system prompt.

**The scenario:** an agent that writes SQL queries across different business verticals in a
large organization. Loading every schema into the prompt upfront would overwhelm the context
window, especially with many verticals or large schemas. Progressive disclosure fixes this by
loading only the relevant schema when it's actually needed.

**What we'll build:** a SQL assistant with two skills (sales analytics and inventory
management). The agent sees lightweight skill *descriptions* in its system prompt, and loads
the full database schema and business logic through a tool call only when relevant.

```mermaid
flowchart TD
    Start([User: Write SQL query<br/>for high-value customers]) --> SystemPrompt[Agent sees skill descriptions:<br/>sales_analytics, inventory_management]
    SystemPrompt --> Decide{Need sales schema}
    Decide --> LoadSkill[load_skill'sales_analytics']
    LoadSkill --> Schema[Schema loaded:<br/>customers, orders tables<br/>+ business logic]
    Schema --> WriteQuery[Agent writes SQL query<br/>using schema knowledge]
    WriteQuery --> Response([Returns valid SQL<br/>following business rules])

    classDef startEnd fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef process fill:#FDF3FF,stroke:#7E65AE,stroke-width:2px,color:#504B5F
    classDef decision fill:#E5F4FF,stroke:#006DDD,stroke-width:2px,color:#030710
    classDef enrichment fill:#EBD0F0,stroke:#885270,stroke-width:2px,color:#441E33
    class Start,Response startEnd
    class SystemPrompt,LoadSkill,WriteQuery process
    class Decide decision
    class Schema enrichment
```

**Why progressive disclosure:** it reduces context usage (only 2-3 relevant skills load per
task, not all of them), lets different teams maintain their own skills independently, scales
to dozens or hundreds of skills without overwhelming context, and keeps the conversation
history simple -- one agent, one thread, no sub-agent orchestration to track.

**Trade-offs worth knowing:** loading a skill on demand adds one extra tool-call round trip
the first time it's needed, and basic implementations rely on prompting (not hard constraints)
to guide skill usage -- enforcing "always load A before B" needs custom logic, shown in
section 6 below.

## Setup

Copy `.env.example` to `.env` and set `OPENAI_API_KEY=sk-...` before running this notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Copy .env.example to .env and add your key.")

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
print("Model ready.")


## 1. Define skills

Each skill has a name, a brief `description` (shown in the system prompt up front), and the
full `content` (loaded on demand). The description stays lightweight; the content can be as
large as the schema actually requires.


In [ ]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str
    description: str
    content: str

SKILLS: list[Skill] = [
    {
        "name": "sales_analytics",
        "description": "Database schema and business logic for sales data analysis including customers, orders, and revenue.",
        "content": """# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from
orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT c.customer_id, c.name, c.customer_tier, SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed' AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
""",
    },
    {
        "name": "inventory_management",
        "description": "Database schema and business logic for inventory tracking including products, warehouses, and stock levels.",
        "content": """# Inventory Management Schema

## Tables

### products
- product_id (PRIMARY KEY)
- product_name
- sku
- category
- unit_cost
- reorder_point (minimum stock level before reordering)
- discontinued (boolean)

### warehouses
- warehouse_id (PRIMARY KEY)
- warehouse_name
- location
- capacity

### inventory
- inventory_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- quantity_on_hand
- last_updated

### stock_movements
- movement_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- movement_type (inbound/outbound/transfer/adjustment)
- quantity (positive for inbound, negative for outbound)
- movement_date
- reference_number

## Business Logic

**Available stock**: quantity_on_hand from inventory table where quantity_on_hand > 0

**Products needing reorder**: Products where total quantity_on_hand across all warehouses is
less than or equal to the product's reorder_point

**Active products only**: Exclude products where discontinued = true unless specifically
analyzing discontinued items

**Stock valuation**: quantity_on_hand * unit_cost for each product

## Example Query

-- Find products below reorder point across all warehouses
SELECT p.product_id, p.product_name, p.reorder_point, SUM(i.quantity_on_hand) as total_stock,
       p.unit_cost, (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
""",
    },
]
print(f"{len(SKILLS)} skills defined.")


## 2. Create the skill-loading tool

One tool loads whichever skill's full content the agent asks for.


In [ ]:
from langchain.tools import tool

@tool
def load_skill(skill_name: str) -> str:
    """Load the full content of a skill into the agent's context.

    Use this when you need detailed information about how to handle a specific
    type of request. This will provide you with comprehensive instructions,
    policies, and guidelines for the skill area.

    Args:
        skill_name: The name of the skill to load (e.g., "sales_analytics", "inventory_management")
    """
    for skill in SKILLS:
        if skill["name"] == skill_name:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"
    available = ", ".join(s["name"] for s in SKILLS)
    return f"Skill '{skill_name}' not found. Available skills: {available}"

print("load_skill tool defined.")


## 3. Build skill-discovery middleware

Custom middleware injects just the lightweight skill *descriptions* into the system prompt --
not the full content -- so the agent knows what's available without paying the context cost
upfront.


In [ ]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """Middleware that injects skill descriptions into the system prompt."""

    tools = [load_skill]  # registered as a class variable

    def __init__(self):
        skills_list = [f"- **{skill['name']}**: {skill['description']}" for skill in SKILLS]
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use the load_skill tool when you need detailed information "
            "about handling a specific type of request."
        )
        new_content = list(request.system_message.content_blocks) + [{"type": "text", "text": skills_addendum}]
        new_system_message = SystemMessage(content=new_content)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

print("SkillMiddleware defined.")


## 4. Create the agent

A checkpointer keeps conversation history across turns, the same way it does in the other
projects.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

sql_agent = create_agent(
    model,
    system_prompt="You are a SQL query assistant that helps users write queries against business databases.",
    middleware=[SkillMiddleware()],
    checkpointer=InMemorySaver(),
)
print("SQL assistant created.")


## 5. Test progressive disclosure


In [ ]:
from langchain_core.utils.uuid import uuid7

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

result = sql_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Write a SQL query to find all customers who made orders over $1000 in the last month"}
        ]
    },
    config,
)
for message in result["messages"]:
    message.pretty_print()


The agent saw only the lightweight skill *description* in its system prompt, recognized this
question needed sales-database knowledge, called `load_skill("sales_analytics")` to pull in
the full schema and business logic, and then used that to write a correct query.

## 6. Advanced: constraining tool use with custom state

You can enforce that certain tools are only usable *after* a specific skill has been loaded --
useful when a tool shouldn't run on guesswork about schema details it hasn't actually seen.

```mermaid
graph LR
    A[write_sql_query called] --> B{Was the matching<br/>skill loaded?}
    B -->|No| C[Return error,<br/>telling agent to load_skill first]
    B -->|Yes| D[Validate and format query]
```

### Track loaded skills in state


In [ ]:
from langchain.agents.middleware import AgentState
from typing_extensions import NotRequired

class CustomState(AgentState):
    skills_loaded: NotRequired[list[str]]

print("CustomState defined.")


### Update `load_skill` to record what it loaded


In [ ]:
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage

@tool
def load_skill_tracked(skill_name: str, runtime: ToolRuntime) -> Command:
    """Load the full content of a skill into the agent's context.

    Use this when you need detailed information about how to handle a specific
    type of request.

    Args:
        skill_name: The name of the skill to load
    """
    for skill in SKILLS:
        if skill["name"] == skill_name:
            skill_content = f"Loaded skill: {skill_name}\n\n{skill['content']}"
            return Command(
                update={
                    "messages": [ToolMessage(content=skill_content, tool_call_id=runtime.tool_call_id)],
                    "skills_loaded": [skill_name],
                }
            )
    available = ", ".join(s["name"] for s in SKILLS)
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"Skill '{skill_name}' not found. Available skills: {available}",
                    tool_call_id=runtime.tool_call_id,
                )
            ]
        }
    )

print("load_skill_tracked defined.")


### Create the constrained tool


In [ ]:
@tool
def write_sql_query(query: str, vertical: str, runtime: ToolRuntime) -> str:
    """Write and validate a SQL query for a specific business vertical.

    This tool helps format and validate SQL queries. You must load the
    appropriate skill first to understand the database schema.

    Args:
        query: The SQL query to write
        vertical: The business vertical (sales_analytics or inventory_management)
    """
    skills_loaded = runtime.state.get("skills_loaded", [])
    if vertical not in skills_loaded:
        return (
            f"Error: You must load the '{vertical}' skill first "
            f"to understand the database schema before writing queries. "
            f"Use load_skill('{vertical}') to load the schema."
        )
    return f"SQL Query for {vertical}:\n\n```sql\n{query}\n```\n\n✓ Query validated against {vertical} schema\nReady to execute against the database."

print("write_sql_query defined -- it refuses to validate a query for a skill that was never loaded.")


### Wire the constrained tool into middleware and rebuild the agent


In [ ]:
class TrackedSkillMiddleware(AgentMiddleware[CustomState]):
    """Middleware that injects skill descriptions and tracks which have been loaded."""

    state_schema = CustomState
    tools = [load_skill_tracked, write_sql_query]

    def __init__(self):
        skills_list = [f"- **{skill['name']}**: {skill['description']}" for skill in SKILLS]
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        skills_addendum = (
            f"\n\n## Available Skills\n\n{self.skills_prompt}\n\n"
            "Use load_skill_tracked to load a skill's schema before calling write_sql_query "
            "for that vertical."
        )
        new_content = list(request.system_message.content_blocks) + [{"type": "text", "text": skills_addendum}]
        new_system_message = SystemMessage(content=new_content)
        return handler(request.override(system_message=new_system_message))

constrained_agent = create_agent(
    model,
    system_prompt="You are a SQL query assistant that helps users write queries against business databases.",
    middleware=[TrackedSkillMiddleware()],
    checkpointer=InMemorySaver(),
)

thread_id_2 = str(uuid7())
config_2 = {"configurable": {"thread_id": thread_id_2}}
result = constrained_agent.invoke(
    {"messages": [{"role": "user", "content": "Write and validate a SQL query to find products below their reorder point"}]},
    config_2,
)
for message in result["messages"]:
    message.pretty_print()


If the agent tried `write_sql_query` before loading the matching skill, it would get the error
message above, prompting it to load the schema first -- the constraint is enforced in code,
not just suggested in the prompt.

## Implementation variations

This tutorial stores skills as in-memory Python dictionaries, loaded through direct tool
calls. A few other shapes are worth knowing about if your skill set grows:

| Dimension | Options |
|---|---|
| **Storage** | In-memory (this project), file system (the Claude Code approach -- skills as directories), remote storage (S3, a database, Notion, an API) |
| **Discovery** | System-prompt listing (this project), file-system scanning, a registry service, a dynamic `list_skills` tool call |
| **Loading strategy** | Single load (this project), paginated for large skills, search-based (grep/read within a skill's content), hierarchical (overview first, then drill in) |

As a rough sizing guide: skills under ~1K tokens can often just live directly in the system
prompt (and benefit from prompt caching); skills in the 1-10K token range are where on-demand
loading like this project starts paying off; skills over ~10K tokens benefit from pagination
or search-based loading rather than a single big load.

## Key takeaways

| Point | Why it matters |
|---|---|
| Descriptions in the prompt, content behind a tool call | Keeps context small until a skill is actually relevant |
| `load_skill(skill_name)` is the entire mechanism | Everything else (tracking, constraints, hierarchy) builds on this one idea |
| Middleware injects descriptions without a separate agent | One agent, one conversation thread, no sub-agent orchestration |
| Custom state can enforce "load before use" | Turns a prompting suggestion into an actual constraint |

**When to use this pattern:** one agent with many possible specializations, where loading
everything upfront would be wasteful, and no need to enforce hard boundaries *between*
specializations (just within how/when they're used). For distinct domains that need their own
dedicated agent and tools, see the Subagents notebook instead.
